In [ ]:
# =============================================================================
# 0) Install dependencies (Notebook-friendly; safe to skip if already installed)
# =============================================================================
# If you're running this as a .py script, comment these lines.
!pip install -q \
  numpy==1.25.2 \
  torch torchvision torchaudio \
  lime shap xgboost captum scikit-learn scikit-image \
  git+https://github.com/jacobgil/pytorch-grad-cam.git \
  matplotlib seaborn tqdm pillow

# =============================================================================
# 1) Imports & Seeding
# =============================================================================
import os
import zipfile
import random
import warnings
import math
import time
import csv
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models
from torchvision.models import (
    ResNet50_Weights, ResNet18_Weights, DenseNet121_Weights,
    EfficientNet_B0_Weights, VGG19_Weights, Swin_V2_T_Weights
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_score, recall_score, f1_score
)

from tqdm import tqdm

# Explainability libraries
from lime import lime_image
from skimage.segmentation import mark_boundaries
from captum.attr import Occlusion
from pytorch_grad_cam import ScoreCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# SHAP and XGBoost (optional for tabular explainability)
import shap
import xgboost as xgb

import seaborn as sns

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED   = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

# =============================================================================
# 3) Hyperparameters
# =============================================================================
IMAGE_SIZE = 256
BATCH_SIZE = 8
VAL_RATIO  = 0.15
MAX_EPOCHS = 5
PATIENCE   = 5
extract_dir= "DataSet/"
FIG_DIR    = "figures_multi"
OUT_DIR    = "outputs"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

COST_PER_GPU_HOUR = 0.0  # set non-zero to show training cost bars (e.g., 0.35)
ROC_POINTS_LIMIT  = 200  # to keep combined ROC plots light

# =============================================================================
# 4) Dataset Loader Setup (Lung Data)
# =============================================================================
class LungDataset(Dataset):
    def __init__(self, root_dir, transform=None, indices=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        self.samples = []
        for idx, cls in enumerate(self.classes):
            folder = os.path.join(root_dir, cls)
            for fname in os.listdir(folder):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    path = os.path.join(folder, fname)
                    self.samples.append((path, idx))
        self.indices = indices if indices is not None else list(range(len(self.samples)))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        path, label = self.samples[self.indices[i]]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        # return path so we can write inference CSVs later
        return img, label, path

# -----------------------------------------------------------------------------
# 6)  Define Transformations (augment for train only)
# -----------------------------------------------------------------------------
train_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.3, hue=0.02),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# -----------------------------------------------------------------------------
# 7)  Point each split to its own sub-folder
# -----------------------------------------------------------------------------
DATA_ROOT = extract_dir
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "valid")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

# -----------------------------------------------------------------------------
# 8)  Build base datasets (no transforms yet)
# -----------------------------------------------------------------------------
train_base = LungDataset(TRAIN_DIR, transform=None)
val_base   = LungDataset(VAL_DIR,   transform=None)
test_base  = LungDataset(TEST_DIR,  transform=None)

NUM_CLASSES = len(train_base.classes)
print(f"✅ Classes ({NUM_CLASSES}): {train_base.classes}")
print(f"📊 Split sizes  →  train: {len(train_base)} | valid: {len(val_base)} | test: {len(test_base)}")

# -----------------------------------------------------------------------------
# 9)  Up-sample the training set to TARGET per class (index-based upsampling)
# -----------------------------------------------------------------------------
TARGET = 1000
class_to_idxs = defaultdict(list)
for i, (_, lbl) in enumerate(train_base.samples):
    class_to_idxs[lbl].append(i)

aug_train_idx = []
for lbl, idxs in class_to_idxs.items():
    needed = TARGET - len(idxs)
    if needed > 0:
        idxs = idxs + random.choices(idxs, k=needed)
    else:
        idxs = random.sample(idxs, TARGET)
    aug_train_idx.extend(idxs)

print(f"🆙  Augmented train size: {len(aug_train_idx)}")

# -----------------------------------------------------------------------------
# 10) Final Datasets & DataLoaders
# -----------------------------------------------------------------------------
train_ds = LungDataset(TRAIN_DIR, train_transform, aug_train_idx)
val_ds   = LungDataset(VAL_DIR,   val_transform)
test_ds  = LungDataset(TEST_DIR,  val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, drop_last=True,
                          pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, drop_last=False,
                          pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                          shuffle=False, drop_last=False,
                          pin_memory=torch.cuda.is_available())

print(f"📦  Train / Val / Test samples: {len(train_loader.dataset)} / {len(val_loader.dataset)} / {len(test_loader.dataset)}")

# =============================================================================
# 9) Positional Encoding (for ViT-style)
# =============================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(0)].unsqueeze(1)

# =============================================================================
# 10) Model Definitions
# =============================================================================
class ResNetThenViT(nn.Module):
    def __init__(self, num_classes, hidden_dim=768, heads=12, mlp_dim=3072, layers=6):
        super().__init__()
        res = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        self.fe = nn.Sequential(*list(res.children())[:-2])
        self.proj = nn.Conv2d(2048, hidden_dim, kernel_size=1)
        self.flatten = nn.Flatten(2)
        self.cls = nn.Parameter(torch.zeros(1, 1, hidden_dim))
        self.pos = PositionalEncoding(hidden_dim)
        enc = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=heads, dim_feedforward=mlp_dim)
        self.tr = nn.TransformerEncoder(enc, num_layers=layers)
        self.mlp = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        x = self.fe(x)
        x = self.proj(x)
        x = self.flatten(x).permute(2, 0, 1)
        cls = self.cls.expand(-1, x.size(1), -1)
        x = torch.cat((cls, x), dim=0)
        x = self.pos(x)
        x = self.tr(x)
        return self.mlp(x[0])

class ViTThenResNet(nn.Module):
    def __init__(self, num_classes, img_size=IMAGE_SIZE, patch_size=16, dim=768, layers=6, heads=12):
        super().__init__()
        assert img_size % patch_size == 0
        self.img_size = img_size
        self.patch_size = patch_size
        self.dim = dim

        self.proj = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_emb = nn.Parameter(torch.zeros(1, (img_size//patch_size)**2 + 1, dim))
        enc = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, dim_feedforward=3072)
        self.tr = nn.TransformerEncoder(enc, num_layers=layers)

        res = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        res.conv1 = nn.Conv2d(dim, 64, kernel_size=7, stride=2, padding=3, bias=False)
        nn.init.kaiming_normal_(res.conv1.weight, mode="fan_out", nonlinearity="relu")
        res.fc = nn.Linear(res.fc.in_features, num_classes)
        self.resnet = res

    def forward(self, x):
        B = x.size(0)
        x = self.proj(x).flatten(2).transpose(1, 2)
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat((cls, x), dim=1) + self.pos_emb
        x = x.transpose(0, 1)
        x = self.tr(x)
        x = x.transpose(0, 1)[:, 1:, :]
        H = W = self.img_size // self.patch_size
        x = x.transpose(1, 2).reshape(B, self.dim, H, W)
        return self.resnet(x)

class SwinThenResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        swin = models.swin_v2_t(weights=Swin_V2_T_Weights.DEFAULT)
        self.fe = swin.features  # may output NHWC depending on version
        dim = swin.head.in_features

        res = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        res.conv1 = nn.Conv2d(dim, 64, kernel_size=7, stride=2, padding=3, bias=False)
        nn.init.kaiming_normal_(res.conv1.weight, mode="fan_out", nonlinearity="relu")
        res.fc = nn.Linear(res.fc.in_features, num_classes)
        self.resnet = res

    def forward(self, x):
        x = self.fe(x)
        # safety: if NHWC, permute to NCHW
        if x.dim() == 4 and x.shape[1] not in (3, 64, 96, 128, 192, 256, 512, 768):
            x = x.permute(0, 3, 1, 2).contiguous()
        return self.resnet(x)

class SwinThenViT(nn.Module):
    def __init__(self, num_classes, layers=6, heads=12):
        super().__init__()
        swin = models.swin_v2_t(weights=Swin_V2_T_Weights.DEFAULT)
        self.fe = swin.features
        dim = swin.head.in_features

        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = PositionalEncoding(dim)
        enc = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, dim_feedforward=3072)
        self.tr = nn.TransformerEncoder(enc, num_layers=layers)
        self.mlp = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, num_classes))

    def forward(self, x):
        x = self.fe(x)
        if x.dim() == 4 and x.shape[1] not in (3, 64, 96, 128, 192, 256, 512, 768):
            x = x.permute(0, 3, 1, 2).contiguous()
        B, C, H, W = x.shape
        x = x.flatten(2).permute(2, 0, 1)   # [HW, B, C]
        cls = self.cls.expand(-1, B, -1)
        x = torch.cat((cls, x), dim=0)
        x = self.pos(x)
        x = self.tr(x)
        return self.mlp(x[0])

class ResNet50Transfer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        m = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        self.model = m
    def forward(self, x):
        return self.model(x)

class DenseNet121Transfer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        m = models.densenet121(weights=DenseNet121_Weights.DEFAULT)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
        self.model = m
    def forward(self, x):
        return self.model(x)

class EfficientNetB0Transfer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        m = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        self.model = m
    def forward(self, x):
        return self.model(x)

class VGG19Transfer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        m = models.vgg19(weights=VGG19_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        self.model = m
    def forward(self, x):
        return self.model(x)

class SwinV2Transfer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        m = models.swin_v2_t(weights=Swin_V2_T_Weights.DEFAULT)
        m.head = nn.Linear(m.head.in_features, num_classes)
        self.model = m
    def forward(self, x):
        return self.model(x)

MODEL_DICT = {
    "resnet_then_vit": ResNetThenViT,
    "vit_then_resnet": ViTThenResNet,
    "swin_then_resnet": SwinThenResNet,
    "swin_then_vit": SwinThenViT,
    "resnet50_transfer": ResNet50Transfer,
    "densenet121_transfer": DenseNet121Transfer,
    "efficientnet_b0_transfer": EfficientNetB0Transfer,
    "vgg19_transfer": VGG19Transfer,
    "swin_v2_transfer": SwinV2Transfer,
}

# =============================================================================
# 11) Training & Evaluation Helpers
# =============================================================================
criterion = nn.CrossEntropyLoss()
CLASS_NAMES = train_ds.classes

def _unpack(batch):
    # supports (imgs, labels) or (imgs, labels, paths)
    if len(batch) == 3:
        imgs, labels, paths = batch
    else:
        imgs, labels = batch
        paths = None
    return imgs, labels, paths

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in tqdm(loader, desc="Train", leave=False):
        imgs, labels, _ = _unpack(batch)
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (preds.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss/total, correct/total

def eval_one_epoch(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Eval", leave=False):
            imgs, labels, _ = _unpack(batch)
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs)
            loss  = criterion(preds, labels)
            total_loss += loss.item() * imgs.size(0)
            correct    += (preds.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
            all_preds.extend(preds.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    unique_labels = sorted(set(all_labels))
    used_classes = [CLASS_NAMES[i] for i in unique_labels]

    report = classification_report(
        all_labels,
        all_preds,
        labels=unique_labels,
        target_names=used_classes,
        zero_division=0
    )

    return total_loss / total, correct / total, report, np.array(all_labels), np.array(all_preds)

def get_probs_labels(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            imgs, labels, _ = _unpack(batch)
            imgs = imgs.to(device)
            logits = model(imgs)
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_labels.extend(labels.numpy())
    return np.concatenate(all_probs, axis=0), np.array(all_labels)

def macro_specificity(y_true, y_pred, num_classes):
    specs = []
    for c in range(num_classes):
        tn = np.sum((y_true != c) & (y_pred != c))
        fp = np.sum((y_true != c) & (y_pred == c))
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specs.append(spec)
    return float(np.mean(specs))

def measure_inference_time(model, loader, n_batches=10):
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches:
                break
            imgs, _, _ = _unpack(batch)
            imgs = imgs.to(device)
            t0 = time.time()
            _ = model(imgs)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            times.append(time.time() - t0)
    if not times: return 0.0
    avg_per_batch = np.mean(times)
    return float(avg_per_batch / imgs.size(0))

def infer_to_csv(model, loader, class_names, save_path):
    """Run inference on loader and save per-image predictions & probabilities."""
    model.eval()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    header = ["filepath", "true_label", "pred_label"] + [f"prob_{c}" for c in class_names]
    rows = []
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference (CSV)", leave=False):
            imgs, labels, paths = _unpack(batch)
            imgs = imgs.to(device)
            logits = model(imgs)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = logits.argmax(1).cpu().numpy()
            lbls  = labels.numpy()
            for pth, y, yhat, probrow in zip(paths, lbls, preds, probs):
                rows.append([pth, class_names[y], class_names[yhat]] + list(map(float, probrow)))
            all_preds.extend(preds); all_labels.extend(lbls)
    with open(save_path, "w", newline="") as f:
        writer = csv.writer(f); writer.writerow(header); writer.writerows(rows)
    acc = (np.array(all_preds) == np.array(all_labels)).mean()
    return float(acc)

# =============================================================================
# 12) Train / Resume All Models + Curves + Capture Metrics for "All Graphs"
# =============================================================================
results = {}
per_model_metrics = {}  # to aggregate for "Proposed Methodology" graphs
roc_train_curves = {}   # model -> (fpr_macro, tpr_macro, auc_macro)
roc_test_curves  = {}

for name, cls in MODEL_DICT.items():
    print(f"\n=== Processing model: {name} ===")
    best_model_path = f"{name}_best.pth"
    checkpoint_path = f"{name}_checkpoint.pth"
    save_dir = os.path.join(FIG_DIR, name)  # figures_model per model
    os.makedirs(save_dir, exist_ok=True)

    model = cls(num_classes=NUM_CLASSES).to(device)
    optimizer = optim.Adam(model.parameters(), lr=3e-5, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                     patience=1, factor=0.5)

    start_epoch = 1
    best_acc = 0.0
    history = {"tr_loss": [], "val_loss": [], "tr_acc": [], "val_acc": []}

    # ---------- resume ----------
    if os.path.isfile(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        if "scheduler_state" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_acc = ckpt["best_acc"]
        history = ckpt.get("history", history)
        print(f"→ Resumed from epoch {ckpt['epoch']} (best_acc={best_acc:.4f})")

    # Already trained enough?
    if start_epoch > MAX_EPOCHS:
        print("→ Training already complete. Skipping training.")
        model.load_state_dict(torch.load(best_model_path, map_location=device))
        results[name] = {"best_val_acc": best_acc, "history": history}
    else:
        # ---------- training ----------
        patience_counter = 0
        train_t0 = time.time()
        for epoch in range(start_epoch, MAX_EPOCHS + 1):

            # optionally freeze backbone for the first epoch of a hybrid model
            if name == "resnet_then_vit" and epoch == 1 and hasattr(model, "fe"):
                for p in model.fe.parameters(): p.requires_grad = False
            else:
                for p in model.parameters():   p.requires_grad = True

            tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer)
            val_loss, val_acc, _, _, _ = eval_one_epoch(model, val_loader)
            scheduler.step(val_loss)

            history["tr_loss"].append(tr_loss)
            history["val_loss"].append(val_loss)
            history["tr_acc"].append(tr_acc)
            history["val_acc"].append(val_acc)

            print(f"Epoch {epoch}/{MAX_EPOCHS}  "
                  f"tr_loss={tr_loss:.4f}  val_loss={val_loss:.4f}  "
                  f"tr_acc={tr_acc:.4f}  val_acc={val_acc:.4f}")

            # Save best (early stopping monitored on val_acc)
            if val_acc > best_acc:
                best_acc = val_acc
                torch.save(model.state_dict(), best_model_path)
                patience_counter = 0
                print(f"  ↳ New best model saved (val_acc={best_acc:.4f})")
            else:
                patience_counter += 1

            # Save checkpoint
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "best_acc": best_acc,
                "history": history,
            }, checkpoint_path)

            if patience_counter >= PATIENCE:
                print("  ↳ Early stopping triggered.")
                break

        train_total_time = time.time() - train_t0
        results[name] = {"best_val_acc": best_acc, "history": history, "train_time_sec": train_total_time}

    # ---------- Load best & collect metrics for "All Graphs" ----------
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()

    # Train/Val/Test evaluation and predictions
    tr_loss, tr_acc, _, tr_labels, tr_preds = eval_one_epoch(model, train_loader)
    v_loss, v_acc, _, v_labels, v_preds = eval_one_epoch(model, val_loader)
    te_loss, te_acc, te_report, te_labels, te_preds = eval_one_epoch(model, test_loader)

    # Probs (for ROC)
    train_probs, train_labels = get_probs_labels(model, train_loader)
    test_probs,  test_labels  = get_probs_labels(model, test_loader)

    # Precision/Recall/F1 (macro for multiclass)
    avg_kw = {"average": "macro"} if NUM_CLASSES > 2 else {}
    prec_test = precision_score(test_labels, te_preds, **avg_kw, zero_division=0)
    rec_test  = recall_score(test_labels, te_preds, **avg_kw, zero_division=0)
    f1_test   = f1_score(test_labels, te_preds, **avg_kw, zero_division=0)
    spec_test = macro_specificity(test_labels, te_preds, NUM_CLASSES)
    sens_test = recall_score(test_labels, te_preds, **avg_kw, zero_division=0)

    # Inference time per sample
    avg_infer_time = measure_inference_time(model, test_loader, n_batches=10)

    # Extra: explicit test-set inference CSV (and infer-accuracy == test accuracy)
    csv_path = os.path.join(OUT_DIR, f"{name}_test_inference.csv")
    infer_acc = infer_to_csv(model, test_loader, CLASS_NAMES, csv_path)
    print(f"📝 Saved per-image test inference CSV to {csv_path} | Inference Acc={infer_acc:.4f}")

    # Training time seconds
    train_time_sec = results[name].get("train_time_sec", np.nan)
    train_cost = (train_time_sec / 3600.0) * COST_PER_GPU_HOUR if not np.isnan(train_time_sec) else 0.0

    # Store per-model metrics
    per_model_metrics[name] = {
        "train_acc": float(tr_acc),
        "test_acc": float(te_acc),
        "precision_macro_test": float(prec_test),
        "recall_macro_test": float(rec_test),
        "f1_macro_test": float(f1_test),
        "specificity_macro_test": float(spec_test),
        "sensitivity_macro_test": float(sens_test),
        "train_time_sec": float(train_time_sec) if not np.isnan(train_time_sec) else 0.0,
        "infer_time_sec_per_sample": float(avg_infer_time),
        "train_cost_usd": float(train_cost),
        "history": results[name]["history"],
        "test_labels": test_labels,
        "test_preds": te_preds,
        "train_labels": tr_labels,
        "train_preds": tr_preds,
        "train_probs": train_probs,
        "test_probs": test_probs,
        "report": te_report
    }

    # Save Confusion Matrices (train/test)
    for split, labels_arr, preds_arr, cmap, tag in [
        ("Test", test_labels, te_preds, "Blues", "test"),
        ("Train", tr_labels,   tr_preds, "Greens", "train")
    ]:
        cm = confusion_matrix(labels_arr, preds_arr, labels=range(NUM_CLASSES))
        plt.figure(figsize=(6,5))
        sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        plt.title(f"Confusion Matrix — {name} ({split})")
        plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"{name}_confusion_{tag}.png"), dpi=200)
        plt.close()

    # Training & Validation Accuracy vs Epochs (per model)
    hist = results[name]["history"]
    epochs_ran = list(range(1, len(hist["tr_acc"]) + 1))
    plt.figure()
    plt.plot(epochs_ran, hist["tr_acc"], label="Train Acc")
    plt.plot(epochs_ran, hist["val_acc"], label="Val Acc")
    plt.ylim(0,1); plt.xlabel("Epoch"); plt.ylabel("Accuracy")
    plt.title(f"{name} — Training & Validation Accuracy vs Epochs")
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{name}_train_val_acc_vs_epochs.png"), dpi=200)
    plt.close()

    # Compute macro ROC curves for combined ROC plots
    def macro_roc(probs, labels):
        n_classes = probs.shape[1]
        y_true_bin = np.eye(n_classes)[labels]
        fpr, tpr, roc_auc = {}, {}, {}
        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], probs[:, i])
            if len(fpr[i]) > ROC_POINTS_LIMIT:
                idx = np.linspace(0, len(fpr[i])-1, ROC_POINTS_LIMIT).astype(int)
                fpr[i], tpr[i] = fpr[i][idx], tpr[i][idx]
            roc_auc[i] = auc(fpr[i], tpr[i])
        all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
        mean_tpr = np.zeros_like(all_fpr)
        for i in range(n_classes):
            mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
        mean_tpr /= n_classes
        return all_fpr, mean_tpr, auc(all_fpr, mean_tpr)

    fpr_tr, tpr_tr, auc_tr = macro_roc(train_probs, train_labels)
    fpr_te, tpr_te, auc_te = macro_roc(test_probs,  test_labels)
    roc_train_curves[name] = (fpr_tr, tpr_tr, auc_tr)
    roc_test_curves[name]  = (fpr_te, tpr_te, auc_te)

print("\n✔️  All models processed, metrics captured, confusion matrices saved.")

# =============================================================================
# 13) Part 1 Graphs — Model Performance Evaluation Metrics (per model)
# =============================================================================
# For each model: (1) Train vs Test Acc, (2) Specificity vs Sensitivity,
# (3) Precision/Recall/F1 (Test macro). Confusion matrices saved per model dir.

for name, m in per_model_metrics.items():
    save_dir = os.path.join(FIG_DIR, name)
    os.makedirs(save_dir, exist_ok=True)

    # (1) Train vs Test Accuracy — per model
    plt.figure()
    plt.bar([f"{name}-Train", f"{name}-Test"], [m["train_acc"], m["test_acc"]])
    plt.ylabel("Accuracy"); plt.ylim(0, 1)
    plt.title("Training Accuracy vs Testing Accuracy (per model)")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{name}_1_train_vs_test_acc_per_model.png"), dpi=200)
    plt.close()

    # (2) Specificity vs Sensitivity — grouped (macro)
    plt.figure()
    x = np.arange(1); width = 0.35
    plt.bar(x - width/2, [m["specificity_macro_test"]], width, label="Specificity")
    plt.bar(x + width/2, [m["sensitivity_macro_test"]], width, label="Sensitivity (Recall)")
    plt.xticks(x, [name]); plt.ylim(0, 1)
    plt.ylabel("Score"); plt.title("Specificity vs Sensitivity (Test, macro)")
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{name}_2_specificity_vs_sensitivity.png"), dpi=200)
    plt.close()

    # (3) Precision / Recall / F1 — Test macro
    plt.figure()
    metrics_names = ["Precision", "Recall", "F1"]
    vals = [m["precision_macro_test"], m["recall_macro_test"], m["f1_macro_test"]]
    plt.bar(metrics_names, vals); plt.ylim(0, 1)
    plt.ylabel("Score"); plt.title("Precision / Recall / F1 (Test, macro)")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{name}_3_prf1_per_model.png"), dpi=200)
    plt.close()

print("📊 Part 1 graphs saved for each model.")

# =============================================================================
# 14) Part 2 Graphs — Proposed Methodology (Model Comparison + Time/Cost)
# =============================================================================
model_names = list(per_model_metrics.keys())
train_accs  = [per_model_metrics[n]["train_acc"] for n in model_names]
test_accs   = [per_model_metrics[n]["test_acc"] for n in model_names]
train_times = [per_model_metrics[n]["train_time_sec"] for n in model_names]
infer_times = [per_model_metrics[n]["infer_time_sec_per_sample"] for n in model_names]
train_costs = [per_model_metrics[n]["train_cost_usd"] for n in model_names]

# (5) Training vs Testing Accuracy — model comparison
x = np.arange(len(model_names)); width = 0.35
plt.figure(figsize=(max(8, len(model_names)*0.8), 5))
plt.bar(x - width/2, train_accs, width, label="Train Acc")
plt.bar(x + width/2, test_accs,  width, label="Test Acc")
plt.xticks(x, model_names, rotation=20, ha='right')
plt.ylim(0, 1); plt.ylabel("Accuracy")
plt.title("Training vs Testing Accuracy — Model Comparison")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "5_train_vs_test_acc_comparison.png"), dpi=200)
plt.close()

# (6) Training Time vs Inference Time — grouped per model
plt.figure(figsize=(max(8, len(model_names)*0.8), 5))
plt.bar(x - width/2, train_times, width, label="Training Time (s)")
plt.bar(x + width/2, infer_times, width, label="Avg Inference Time / sample (s)")
plt.xticks(x, model_names, rotation=20, ha='right')
plt.ylabel("Seconds"); plt.title("Training Time vs Inference Time")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "6_time_train_vs_infer.png"), dpi=200)
plt.close()

# (7) Training Cost per model
plt.figure(figsize=(max(8, len(model_names)*0.8), 5))
plt.bar(model_names, train_costs)
plt.xticks(rotation=20, ha='right')
plt.ylabel("Cost (USD)")
plt.title("Training Cost per Model")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "7_training_cost_per_model.png"), dpi=200)
plt.close()

# (8) Training & Validation Accuracy vs Epochs – montage (optional)
plt.figure(figsize=(10,6))
for name in model_names:
    hist = per_model_metrics[name]["history"]
    epochs = list(range(1, len(hist["val_acc"])+1))
    if len(epochs) == 0: continue
    plt.plot(epochs, hist["val_acc"], label=f"{name} (Val)")
plt.ylim(0,1); plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.title("Validation Accuracy vs Epochs — All Models")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "8_val_acc_vs_epochs_all_models.png"), dpi=200)
plt.close()

# (9) ROC Curve — Training (one plot with all models)
plt.figure(figsize=(7,6))
for name, (fpr, tpr, auc_val) in roc_train_curves.items():
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc_val:.3f})")
plt.plot([0,1], [0,1], linestyle="--")
plt.xlim([0,1]); plt.ylim([0,1.05])
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Training (macro-average)")
plt.legend(fontsize=8, loc="lower right"); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "9_roc_training_all_models.png"), dpi=200)
plt.close()

# (10) ROC Curve — Testing (one plot with all models)
plt.figure(figsize=(7,6))
for name, (fpr, tpr, auc_val) in roc_test_curves.items():
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc_val:.3f})")
plt.plot([0,1], [0,1], linestyle="--")
plt.xlim([0,1]); plt.ylim([0,1.05])
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Testing (macro-average)")
plt.legend(fontsize=8, loc="lower right"); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "10_roc_testing_all_models.png"), dpi=200)
plt.close()

print(f"📁 All graphs saved to: {FIG_DIR}/  | Per-image CSVs in: {OUT_DIR}/")

# =============================================================================
# 15) XAI for ALL models (Occlusion, LIME, ScoreCAM with proper targets)
# =============================================================================
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406], device=tensor.device).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225], device=tensor.device).view(3, 1, 1)
    img = tensor * std + mean
    return torch.clamp(img, 0, 1).permute(1, 2, 0).cpu().numpy()

def predict_numpy(images, model):
    batch = [val_transform(Image.fromarray(img)) for img in images]
    batch = torch.stack(batch).to(device)
    with torch.no_grad():
        return torch.softmax(model(batch), dim=1).cpu().numpy()

for name, cls in MODEL_DICT.items():
    ckpt = f"{name}_best.pth"
    if not os.path.exists(ckpt):
        print(f"❌ Skipping XAI for {name} – checkpoint not found.")
        continue

    print(f"\n>>> 🔍 XAI for Model: {name}")
    model = cls(num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()

    batch = next(iter(val_loader))
    imgs, labels, _ = _unpack(batch)
    imgs, labels = imgs.to(device), labels.to(device)
    pred_logits = model(imgs)
    pred_cls = pred_logits[0].argmax().item()

    orig_img_np = denormalize(imgs[0])

    # Original
    plt.figure(figsize=(4,4))
    plt.imshow(orig_img_np)
    plt.title(f"Original (Predicted: {CLASS_NAMES[pred_cls]})")
    plt.axis("off")
    plt.show()

    # Occlusion
    occ = Occlusion(model)
    attr = occ.attribute(
        imgs[0].unsqueeze(0),
        strides=(3, 15, 15),
        sliding_window_shapes=(3, 31, 31),
        baselines=0,
        target=pred_cls
    )
    attr_mask = attr.sum(dim=1, keepdim=True)
    upsampled = F.interpolate(attr_mask, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
    heatmap = upsampled.squeeze().cpu().numpy()
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() + 1e-8)
    colored_mask = plt.get_cmap("jet")(heatmap)[:, :, :3]
    overlay = (1.0 - 0.4) * orig_img_np + 0.4 * colored_mask

    plt.figure(figsize=(4,4))
    plt.imshow(overlay)
    plt.title("Occlusion Overlay")
    plt.axis("off")
    plt.show()

    # LIME
    img_for_lime = (orig_img_np * 255).astype("uint8")
    explainer = lime_image.LimeImageExplainer()
    exp = explainer.explain_instance(
        img_for_lime,
        lambda im: predict_numpy(im, model),
        labels=(pred_cls,),
        hide_color=0,
        num_samples=200
    )
    temp, mask = exp.get_image_and_mask(
        label=pred_cls,
        positive_only=True,
        num_features=5,
        hide_rest=False
    )
    plt.figure(figsize=(4,4))
    plt.imshow(mark_boundaries(temp / 255.0, mask))
    plt.title("LIME Explanation")
    plt.axis("off")
    plt.show()

    # ScoreCAM (rough target layer guesses per model)
    if name == "resnet50_transfer":
        target_layer = model.model.layer4[-1]
    elif name == "densenet121_transfer":
        target_layer = model.model.features[-1]
    elif name == "efficientnet_b0_transfer":
        target_layer = model.model.features[-1]
    elif name == "vgg19_transfer":
        target_layer = model.model.features[-1]
    elif name == "swin_v2_transfer":
        target_layer = model.model.features[-1]
    elif name == "resnet_then_vit":
        target_layer = list(model.fe.children())[-1]
    elif name == "vit_then_resnet":
        target_layer = model.resnet.layer4[-1]
    elif name == "swin_then_resnet":
        target_layer = model.resnet.layer4[-1]
    elif name == "swin_then_vit":
        target_layer = list(model.fe.children())[-1]
    else:
        target_layer = list(model.children())[-2]

    with ScoreCAM(model=model, target_layers=[target_layer]) as cam:
        gcam = cam(input_tensor=imgs[0].unsqueeze(0), targets=[ClassifierOutputTarget(pred_cls)])[0]

    overlay2 = show_cam_on_image(orig_img_np, gcam, use_rgb=True)

    plt.figure(figsize=(4,4))
    plt.imshow(overlay2)
    plt.title("ScoreCAM Overlay")
    plt.axis("off")
    plt.show()

# =============================================================================
# 16) Confusion Matrix Helper (Reusable) + Run For All Models (Val & Test)
# =============================================================================
def plot_confusion_matrix(model, loader, class_names, title="Confusion Matrix"):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            imgs, labels, _ = _unpack(batch)
            imgs = imgs.to(device); labels = labels.to(device)
            preds = model(imgs).argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    cm = confusion_matrix(all_labels, all_preds, labels=range(len(class_names)))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title); plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout(); plt.show(); plt.close()
    return cm

for name, cls in MODEL_DICT.items():
    ckpt_path = f"{name}_best.pth"
    if not os.path.exists(ckpt_path):
        print(f"❌ Skipping {name} – checkpoint not found.")
        continue
    print(f"\n✅ Confusion Matrices (val/test) for: {name}")
    model = cls(num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    _ = plot_confusion_matrix(model, val_loader, CLASS_NAMES, title=f"Confusion Matrix – {name} (Val)")
    _ = plot_confusion_matrix(model, test_loader, CLASS_NAMES, title=f"Confusion Matrix – {name} (Test)")

# =============================================================================
# 17) Test-set Evaluation for All Models (Robust Version)
# =============================================================================
test_results = {}
for name, cls in MODEL_DICT.items():
    best_path = f"{name}_best.pth"
    if not os.path.isfile(best_path):
        print(f"❌ Skipping {name}: checkpoint not found.")
        continue
    print(f"\n🧪 Evaluating on Test Set: {name}")
    try:
        model = cls(num_classes=NUM_CLASSES).to(device)
        model.load_state_dict(torch.load(best_path, map_location=device))
        model.eval()
        test_loss, test_acc, test_report, _, _ = eval_one_epoch(model, test_loader)
        print(f"✅ Accuracy: {test_acc:.4f}")
        print(test_report)
        test_results[name] = {
            "test_acc":   float(test_acc),
            "test_loss":  float(test_loss),
            "report":     test_report,
        }
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception as e:
        print(f"⚠️ Error evaluating {name}: {e}")
        continue

print("\n🎯 All completed models evaluated on the test set.")
